In [5]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/entbappy/Kidney-Disease-Classification-MLflow-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "entbappy"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "6824692c47a369aa6f9eac5b10041d5c8edbcef0"


In [6]:
import tensorflow as tf
from pathlib import Path

MODEL_PATH = Path("artifacts/training/model.h5")

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [7]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: Dict
    mlflow_uri: str
    params_image_size: List[int]
    params_batch_size: int


In [8]:
from kidney_disease_classification.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from kidney_disease_classification.utils.common import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_evaluation_config(self) -> EvaluationConfig:
        return EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path(self.config.data_ingestion.unzip_dir) / "kidney-ct-scan-image",
            mlflow_uri=os.environ["MLFLOW_TRACKING_URI"],
            all_params=dict(self.params),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )


CONFIG: C:\satvik\legal\dl-projects\kidney-disease-classification\config\config.yaml
PARAMS: C:\satvik\legal\dl-projects\kidney-disease-classification\params.yaml


In [9]:
import mlflow
import mlflow.keras
from urllib.parse import urlparse

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:2],  # ✅ FIX
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    def load_model(self):
        return tf.keras.models.load_model(self.config.path_of_model)

    def evaluation(self):
        self.model = self.load_model()
        self._valid_generator()

        self.score = self.model.evaluate(self.valid_generator, verbose=1)
        self.save_score()

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }
        save_json(Path("scores.json"), scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {
                    "loss": self.score[0],
                    "accuracy": self.score[1]
                }
            )

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model,
                    "model",
                    registered_model_name="KidneyDiseaseVGG16"
                )
            else:
                mlflow.keras.log_model(self.model, "model")


c:\Users\sharm\miniconda3\envs\dlproj\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()

    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

    print("✅ Evaluation completed and logged to MLflow")

except Exception as e:
    raise e


[2026-01-23 23:17:18,433: INFO: common: YAML file loaded successfully: C:\satvik\legal\dl-projects\kidney-disease-classification\config\config.yaml]
[2026-01-23 23:17:18,436: INFO: common: YAML file loaded successfully: C:\satvik\legal\dl-projects\kidney-disease-classification\params.yaml]
[2026-01-23 23:17:18,439: INFO: common: Directory created at: artifacts]
Found 139 images belonging to 2 classes.
18/18 [==============================] - 33s 2s/step - loss: 0.0599 - accuracy: 0.9928
[2026-01-23 23:17:51,778: INFO: common: JSON file saved at: scores.json]


2026/01/23 23:17:56 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-01-23 23:18:00,100: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\sharm\AppData\Local\Temp\tmp_fkagon_\model\data\model\assets
[2026-01-23 23:18:02,015: INFO: builder_impl: Assets written to: C:\Users\sharm\AppData\Local\Temp\tmp_fkagon_\model\data\model\assets]


c:\Users\sharm\miniconda3\envs\dlproj\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Successfully registered model 'KidneyDiseaseVGG16'.
2026/01/23 23:19:56 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: KidneyDiseaseVGG16, version 1
Created version '1' of model 'KidneyDiseaseVGG16'.


✅ Evaluation completed and logged to MLflow
